# Notebook 07 — Checagem de evidência e recusa

Quarto notebook do **pipeline RAG**. A busca **sempre** devolve um top-k, mesmo
quando nada ali responde a pergunta (notebook 02). O requisito 4 do TC4 pede o
contrário: **sem evidência, o sistema recusa**. Entre a busca e a geração entram duas
barreiras:

1. **Similaridade** (sem LLM): reviews abaixo de `LIMIAR_EVIDENCIA` não contam.
2. **Juiz LLM** (uma chamada): lê as candidatas e diz quais respondem de fato.

Se sobrarem menos de `MIN_EVIDENCIAS` reviews, o RAG recusa.

| Seção | Tema | Função em `src/voc_rag` | Aula de origem |
|---|---|---|---|
| 0 | Setup | — | — |
| 1 | Três tipos de pergunta | `avaliacao.PERGUNTAS_*` | notebook 03 |
| 2 | Barreira 1: só a similaridade | `evidencia.checar_evidencia` | IA Gen 02 · Requisito 4 do TC4 |
| 3 | O contexto delimitado | `evidencia.formatar_contexto` | IA Gen 06 (slide 16) · IA Gen 03 |
| 4 | Barreira 2: o juiz LLM | `evidencia.criar_juiz` | LangChain 02 |
| 5 | As duas juntas: quem responde e quem recusa | `evidencia.checar_evidencia(juiz=...)` | — |

> Pré-requisitos: notebooks 01 a 06 e o `.env` com a chave do LLM.

In [ ]:
import inspect

import pandas as pd

from voc_rag import avaliacao, busca, config, evidencia, indice, llm, multiquery
from voc_rag.preparacao import carregar_base

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)


def mostrar_codigo(objeto):
    """Imprime o código-fonte de uma função ou classe do src/, para lermos junto com o resultado."""
    print(inspect.getsource(objeto))


documentos = carregar_base()
indice_vetorial, _ = indice.carregar_indice()
buscador = busca.Buscador(documentos, indice_vetorial, indice.carregar_modelo())
modelo_chat = llm.criar_llm()
gerador = multiquery.criar_gerador_variacoes(modelo_chat)
juiz = evidencia.criar_juiz(modelo_chat)
print(f"LIMIAR_EVIDENCIA = {config.LIMIAR_EVIDENCIA} | MIN_EVIDENCIAS = {config.MIN_EVIDENCIAS} "
      f"| K_CONTEXTO = {config.K_CONTEXTO}")

## Seção 1 — Três tipos de pergunta

- **dentro:** as 10 perguntas de checagem do notebook 03 — devem ser respondidas;
- **fora:** fora do tema ("capital da França") — devem ser recusadas;
- **sem evidência:** do domínio, mas sem resposta numa base de 2016–2018 ("entrega por
  drone", "metaverso") — devem ser recusadas. É o caso difícil: o vocabulário
  ("entrega", "cliente", "compra") puxa a similaridade para cima.

Para cada pergunta, recuperamos o top-10 com o Multi-Query do notebook 06. Os
resultados ficam guardados para as seções seguintes não repetirem as chamadas.

> O Multi-Query faz uma chamada ao LLM por pergunta (18 no total), espaçadas pelo
> limitador: esta célula leva uns três minutos.

In [ ]:
GRUPOS = {
    "dentro": [p for p, _ in avaliacao.PERGUNTAS_CHECAGEM],
    "fora": avaliacao.PERGUNTAS_FORA_DE_ESCOPO,
    "sem evidência": avaliacao.PERGUNTAS_SEM_EVIDENCIA,
}
recuperados = {}
for grupo, perguntas in GRUPOS.items():
    for pergunta in perguntas:
        consultas, _ = multiquery.gerar_consultas(gerador, pergunta)
        recuperados[pergunta] = (grupo, multiquery.buscar_multi_query(buscador, consultas,
                                                                     k=config.K_CONTEXTO))
print(f"{len(recuperados)} perguntas recuperadas")

## Seção 2 — Barreira 1: só a similaridade

`checar_evidencia` sem juiz conta quantas reviews do top-10 têm similaridade acima do
limiar. Repare no grupo "sem evidência": se a similaridade bastasse, ele seria recusado
como o grupo "fora". Não é o que acontece?

In [ ]:
mostrar_codigo(evidencia.checar_evidencia)

In [ ]:
linhas = []
for pergunta, (grupo, resultado) in recuperados.items():
    d = evidencia.checar_evidencia(resultado, pergunta)          # sem juiz
    linhas.append({"grupo": grupo, "pergunta": pergunta, "sim_top1": d.similaridade_top1,
                   "acima_do_limiar": d.acima_do_limiar, "responde": d.tem_evidencia})
so_similaridade = pd.DataFrame(linhas)
so_similaridade

In [ ]:
so_similaridade.groupby("grupo")[["sim_top1", "responde"]].mean().round(2)

## Seção 3 — O contexto delimitado *(IA Gen aula 06, slide 16 · IA Gen aula 03)*

O juiz (e, na etapa 5, o gerador) lê as reviews. Uma review é texto de um cliente e
pode conter "ignore as instruções" — a injeção indireta do notebook 04, Seção 6. Por
isso cada review entra numa tag `<review id=...>` com os metadados, o texto vai com
`<` e `>` escapados (a review não consegue fechar a tag) e reviews com sinal de injeção
recebem `suspeita="sim"`. O prompt diz que o conteúdo das tags é dado, nunca instrução.

In [ ]:
mostrar_codigo(evidencia.formatar_contexto)
_, exemplo = recuperados[avaliacao.PERGUNTAS_CHECAGEM[0][0]]
print(evidencia.formatar_contexto(exemplo.head(3)))

## Seção 4 — Barreira 2: o juiz LLM *(LangChain aula 02)*

Um LLM como juiz, com saída estruturada (`JulgamentoEvidencia`: os ids relevantes e o
motivo). Três cuidados:

- o prompt pede rigor: elogio genérico ("ótimo, recomendo") **não** é evidência;
- o juiz só pode citar ids das candidatas; id inventado é **descartado** e vira aviso;
- se o juiz falhar, o RAG **recusa** — sem checagem, melhor não responder do que inventar.

In [ ]:
print(evidencia.PROMPT_JUIZ.messages[0].prompt.template)

In [ ]:
for pergunta in [avaliacao.PERGUNTAS_CHECAGEM[0][0], avaliacao.PERGUNTAS_SEM_EVIDENCIA[0]]:
    _, resultado = recuperados[pergunta]
    d = evidencia.checar_evidencia(resultado, pergunta, juiz=juiz)
    print(f"{pergunta!r}\n   responde={d.tem_evidencia} | juiz aprovou {d.aprovadas_pelo_juiz} "
          f"de {d.acima_do_limiar} | {d.motivo}\n")

## Seção 5 — As duas barreiras juntas

A mesma bateria da Seção 2, agora com o juiz. O que se espera: "dentro" responde,
"fora" e "sem evidência" recusam. A última coluna compara com a Seção 2.

> Mais uma chamada ao LLM por pergunta com candidatas acima do limiar (~2 minutos).

In [ ]:
linhas = []
for pergunta, (grupo, resultado) in recuperados.items():
    d = evidencia.checar_evidencia(resultado, pergunta, juiz=juiz)
    linhas.append({"grupo": grupo, "pergunta": pergunta, "acima_do_limiar": d.acima_do_limiar,
                   "aprovadas_juiz": d.aprovadas_pelo_juiz, "responde": d.tem_evidencia,
                   "motivo": d.motivo[:90]})
com_juiz = pd.DataFrame(linhas)
com_juiz["mudou_com_juiz"] = com_juiz["responde"] != so_similaridade["responde"]
com_juiz

In [ ]:
esperado = {"dentro": True, "fora": False, "sem evidência": False}
for nome, tabela in (("só similaridade", so_similaridade), ("similaridade + juiz", com_juiz)):
    acerto = (tabela["responde"] == tabela["grupo"].map(esperado)).groupby(tabela["grupo"]).mean()
    print(f"{nome:20s}", acerto.round(2).to_dict())

## Resumo

- A **similaridade** recusa barato o que está fora do tema, mas não o que é do domínio
  sem resposta na base.
- O **juiz LLM** lê as candidatas e aprova só as que respondem; com menos de
  `MIN_EVIDENCIAS`, o RAG recusa com o motivo registrado em `Decisao`.
- As reviews entram **delimitadas e escapadas**; id inventado pelo juiz é descartado;
  juiz com erro = recusa.
- `MIN_EVIDENCIAS = 2` é uma escolha de negócio: com uma review só, "os clientes
  reclamam de X?" vira anedota. Ajuste em `config.py` se a análise pedir outra coisa.

**Próxima etapa:** a geração. Com as evidências aprovadas, o LLM escreve a resposta num
schema Pydantic que exige citar os `review_id` — e o código confere que cada citação
está entre as evidências.